# 02 - Nettoyage et consolidation des données

Pipeline de préparation :
1. Suppression justifiée des doublons
2. Typage temporel et harmonisation de la séniorité
3. Regroupement thématique des intitulés métiers
4. Jointure géographique et contrôle de volumétrie
5. Export dans data/processed/

In [1]:
import os
import pandas as pd

# Chemins relatifs stricts
path_raw = os.path.join("..", "data", "raw")
path_processed = os.path.join("..", "data", "processed")

df = pd.read_csv(os.path.join(path_raw, "jobs_in_data.csv"))
df_geo = pd.read_csv(os.path.join(path_raw, "cost_of_living.csv"))

initial_rows = len(df)
print(f"Lignes initiales : {initial_rows}")

Lignes initiales : 9355


## 1. Traitement des doublons stricts
Suppression des doublons identifiés dans l'exploration initiale.

In [2]:
df = df.drop_duplicates()
dropped = initial_rows - len(df)
print(f"Doublons supprimés : {dropped} ({dropped / initial_rows:.1%})")
print(f"Lignes restantes : {len(df)}")

Doublons supprimés : 4014 (42.9%)
Lignes restantes : 5341


## 2. Typage temporel et harmonisation de la séniorité
Conversion de l'année en datetime et mapping explicite des niveaux.

In [3]:
df["work_year_dt"] = pd.to_datetime(df["work_year"].astype(str), format="%Y")

level_map = {
    "Entry-level": "Junior",
    "Mid-level": "Intermédiaire",
    "Senior": "Senior",
    "Executive": "Lead / Exec"
}
df["experience_level_fr"] = df["experience_level"].map(level_map).fillna(df["experience_level"])
print(df["experience_level_fr"].value_counts())

experience_level_fr
Senior           3444
Intermédiaire    1274
Junior            400
Lead / Exec       223
Name: count, dtype: int64


## 3. Standardisation des intitulés métiers
Regroupement en 4 familles principales : IA & Machine Learning, Data Science, Data Engineering et Data Analytics.

In [4]:
def categorize_role(t):
    t = str(t).lower()
    if any(k in t for k in ["machine learning", "ml", "ai", "artificial intelligence", "deep learning"]):
        return "IA & Machine Learning"
    elif any(k in t for k in ["data engineer", "etl", "database", "pipeline"]):
        return "Data Engineering"
    elif any(k in t for k in ["data scientist", "research", "applied scientist"]):
        return "Data Science"
    elif any(k in t for k in ["analyst", "analytics", "bi", "business intelligence"]):
        return "Data Analytics"
    return "Autre Data"

df["job_group"] = df["job_title"].apply(categorize_role)
print(df["job_group"].value_counts())

job_group
Data Science             1446
Data Analytics           1263
Data Engineering         1149
IA & Machine Learning     883
Autre Data                600
Name: count, dtype: int64


## 4. Jointure géographique et contrôle de volumétrie
Enrichissement avec contrôle strict du nombre de lignes.

In [5]:
cols_geo = ["CLDR display name", "Continent", "Sub-region Name"]
geo_clean = df_geo[cols_geo].dropna(subset=["CLDR display name"]).drop_duplicates(subset=["CLDR display name"])

rows_before = len(df)
df = df.merge(geo_clean, left_on="employee_residence", right_on="CLDR display name", how="left")

print(f"Contrôle : {rows_before} avant -> {len(df)} après (écart : {len(df) - rows_before})")
print("\nRépartition par continent :")
print(df["Continent"].value_counts(dropna=False))

Contrôle : 5341 avant -> 5341 après (écart : 0)

Répartition par continent :
Continent
NaN    4829
EU      358
AS       58
SA       47
AF       28
OC       21
Name: count, dtype: int64


## 5. Export de la base propre
Sauvegarde dans data/processed/cleaned_salaries.csv.

In [6]:
output_path = os.path.join(path_processed, "cleaned_salaries.csv")
df.to_csv(output_path, index=False)
print(f"Export terminé : {output_path}")
print(f"Dimensions finales : {df.shape[0]} lignes, {df.shape[1]} colonnes")

Export terminé : ../data/processed/cleaned_salaries.csv
Dimensions finales : 5341 lignes, 18 colonnes
